# RAER — stages 01-05: label, aggregate, review, compare

Run **`RAER_00_audit_colab.ipynb` first**. Its findings are what this notebook assumes:
2,649 annotated clips (20 extra mp4 dropped), label map `1 neutrality, 2 enjoyment,
3 confusion, 4 fatigue, 5 distraction`, and a **shipped split that leaks 93.6% of
subjects** — so we build our own.

Flow: **download → split → frames → label → aggregate → review → compare**.

Media stays on the runtime disk; only derived artifacts go to Drive. Nothing derived
from RAER is published — see `DATA_PROVENANCE.md`.

## 1. Setup

In [ ]:
# Gets RAER-external + LLMSTU-pipeline onto the runtime.
# Tries GitHub first; falls back to a zip in Drive, so this works before the
# RAER-external repo exists on GitHub.
RAER_REPO   = "vaelkokach/RAER-external"
LLMSTU_REPO = "vaelkokach/LLMSTU-pipeline"
DRIVE_ZIP   = "/content/drive/MyDrive/RAER/RAER-external.zip"   # fallback

import os, sys, subprocess, zipfile
try:
    from google.colab import userdata
    def _secret(n):
        try: return userdata.get(n)
        except Exception: return None
except Exception:
    def _secret(n): return os.environ.get(n)

gh = _secret("GITHUB_TOKEN")

def have(name): return os.path.isdir(f"/content/{name}")

def clone(repo):
    name = repo.split("/")[-1]
    if have(name):
        return True
    auth = f"{gh}@" if gh else ""
    r = subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{repo}.git",
                        f"/content/{name}"], capture_output=True, text=True)
    if r.returncode:
        print(f"[setup] clone {repo} failed: {r.stderr.strip().splitlines()[-1:] or r.returncode}")
    return r.returncode == 0

clone(LLMSTU_REPO)
if not clone(RAER_REPO):
    if not os.path.isdir("/content/drive"):
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception:
            pass
    if os.path.exists(DRIVE_ZIP):
        print(f"[setup] falling back to {DRIVE_ZIP}")
        with zipfile.ZipFile(DRIVE_ZIP) as z:
            z.extractall("/content")
    else:
        raise SystemExit(
            "RAER-external is not on GitHub and no zip at " + DRIVE_ZIP + ".\n"
            "Either create the repo and push, or drop RAER-external.zip in "
            "MyDrive/RAER/ and re-run this cell.")

assert have("RAER-external") and have("LLMSTU-pipeline"), "setup failed"
sys.path[:0] = ["/content/RAER-external", "/content/LLMSTU-pipeline"]
os.chdir("/content/RAER-external")
LLMSTU = "/content/LLMSTU-pipeline"
try:
    print("llmstu-pipeline @", subprocess.check_output(
        ["git", "-C", LLMSTU, "rev-parse", "--short", "HEAD"], text=True).strip())
except Exception:
    pass

In [ ]:
!pip install -q kaggle "transformers>=4.57.0" accelerate "huggingface_hub>=0.35.0" \
    pandas pyarrow pyyaml pillow numpy opencv-python-headless
import torch; print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/RAER"); DRIVE.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/raer");               LOCAL.mkdir(parents=True, exist_ok=True)
WORK  = Path("/content/RAER-external/work"); WORK.mkdir(parents=True, exist_ok=True)

os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME") or ""
os.environ["KAGGLE_KEY"]      = _secret("KAGGLE_KEY") or ""
os.environ["HF_TOKEN"]        = _secret("HF_TOKEN") or ""
DATASET = "bearmn/raer-video-emotion-dataset"
!df -h /content | tail -1

## 2. Download the data

~100 GB unzipped. `--unzip` streams so it does not need 2x the space, but check the
free disk above first: the runtime starts with ~47 GB already used.

In [ ]:
RAER_ROOT = LOCAL / "RAER"
if not RAER_ROOT.exists():
    !kaggle datasets download -d {DATASET} -p {LOCAL} --unzip
!du -sh {RAER_ROOT} 2>/dev/null; ls {RAER_ROOT}

## 3. Subject-disjoint split

The shipped split shares 132 of 141 subjects between train and test. This builds a
replacement that keeps every clip of a subject on one side while matching the class
balance. Expect ~2,122 / 527 and a max class delta under 0.5pp.

In [ ]:
!python scripts/06_split.py --annotations {LOCAL} --out {WORK}/split
!cp -r {WORK}/split {DRIVE}/ 2>/dev/null; echo "split copied to Drive"

## 4. Stage 01 — sample frames

8 uniform frames per clip, materialised flat with unique basenames so
`llmstu.caption.run` resolves them. ~21k frames, resumable.

Run a `--limit` pass first and look at the frames before committing to the full set.

In [ ]:
!python scripts/01_frames.py --root {RAER_ROOT} --out {WORK} --limit 20
import glob
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob(str(WORK / "frames" / "*.jpg")))[:6]:
    display(IPImage(p, width=260))

In [ ]:
# full pass
!python scripts/01_frames.py --root {RAER_ROOT} --out {WORK}

## 5. Stage 02 — pseudo-label with the LLMSTU labeler

Calls `llmstu.caption.run` directly. It aborts if this repo's caption settings have
drifted from LLMSTU-pipeline's locked config, so the two tasks stay comparable.

**Smoke-test with `--limit 96` first**: check the JSON parses, the fields are
populated, and throughput is what you expect before starting ~21k frames.

In [ ]:
!python scripts/02_label.py --llmstu {LLMSTU} --out {WORK} --limit 96
import json, pandas as pd
rows = [json.loads(l) for l in open(WORK / "frame_labels.jsonl")]
print(len(rows), "frames labeled")
pd.DataFrame(rows)[["clip_stem", "raer_label", "activity", "gaze_direction",
                    "engagement_level", "phone_visible", "model_confidence"]].head(12)

In [ ]:
# full pass — resumable, so just re-run this cell if the runtime drops
!python scripts/02_label.py --llmstu {LLMSTU} --out {WORK}
!cp {WORK}/frame_labels.jsonl {WORK}/run_provenance.yaml {DRIVE}/

## 6. Stage 03 — aggregate frames to clips

In [ ]:
!python scripts/03_aggregate.py --llmstu {LLMSTU} --out {WORK}
!cp {WORK}/clip_labels.jsonl {DRIVE}/

## 7. Stage 04 — build the human-review sample

200 clips stratified over RAER's label (so confusion, at 37 clips dataset-wide, is
represented) plus a hard subset from the lowest mean confidence. Download the zip and
open `index.html` locally — it is self-contained.

In [ ]:
!python scripts/04_golden.py --llmstu {LLMSTU} --out {WORK} --labeling ./labeling
!cd labeling && zip -qr /content/raer_review.zip index.html data.json
!cp /content/raer_review.zip {DRIVE}/
from google.colab import files
files.download("/content/raer_review.zip")

### Review it

For each clip make **two independent judgements**:

1. **Does RAER's label fit?** agree / wrong (then pick a better one) / unclear
2. **Are our cue labels right?** correct any that are wrong

`v` accepts everything and advances, arrow keys move, progress saves in the browser.
Click **Export golden.json** when done, then upload it below.

In [ ]:
from google.colab import files
up = files.upload()   # choose your exported golden.json
import shutil; shutil.move(list(up)[0], "labeling/golden.json")
print("saved labeling/golden.json")

## 8. Stage 05 — the three-way comparison

**A** coarse agreement, read as kappa rather than accuracy (65% of clips are
neutrality, so accuracy flatters). **B** which visible cues carry each RAER class, by
lift over the corpus baseline. **C** human adjudication, split by sample group.

In [ ]:
!python scripts/05_compare.py --llmstu {LLMSTU} --golden labeling/golden.json
!cp -r {WORK}/compare {DRIVE}/
from IPython.display import HTML
HTML(open(WORK / "compare" / "comparison_report.html").read())

## 9. Carrying this into the thesis

- Numbers are **provisional** until the official copy replaces the Kaggle mirror. Set
  `source.kind: official` in `config.yaml` and re-run.
- Any comparison with the paper's published UAR must state that **their split is not
  subject-disjoint and ours is**, so the two are not directly comparable.
- Confusion has 37 clips dataset-wide. Report it with its n, or fold it into a coarser
  grouping — never a bare per-class score.
- `work/compare/comparison.json` plus `run_provenance.yaml` pin exactly which labeler
  commit and config produced each figure.